# Pruebas para la app de Streamlit
Revisar códigos y rangos de las variables, entrenar un modelo provisional
y probar las funciones antes de pasarlas a app/.

In [21]:
from ses.pipeline import cargar_entrenamiento, NUMERICAS, NOMINALES, BLOQUES

X, y, cv = cargar_entrenamiento()
X.shape

(15728, 32)

In [22]:
for col in NOMINALES:
    print("=====", col)
    print(X[col].value_counts(dropna=False).sort_index())

===== sexo_jefe
sexo_jefe
1    7505
2    8223
Name: count, dtype: int64
===== tipo_vivienda
tipo_vivienda
1    7555
2    7897
3     254
5      18
6       4
Name: count, dtype: int64
===== material_paredes
material_paredes
1    15052
2       99
3       38
4       91
5      329
6        4
7       40
8       74
9        1
Name: count, dtype: int64
===== material_pisos
material_pisos
1      237
2     2821
3      318
4    12251
5       24
6       66
7       11
Name: count, dtype: int64
===== estrato
estrato
0.0     215
1.0    5073
2.0    5252
3.0    3711
4.0     889
5.0     382
6.0     190
7.0      16
Name: count, dtype: int64
===== tipo_sanitario
tipo_sanitario
1    14263
2      972
3      392
4        9
5       50
6       42
Name: count, dtype: int64
===== lugar_cocina
lugar_cocina
1    12834
2      228
3     2177
4      205
5      111
6      173
Name: count, dtype: int64
===== combustible_cocina
combustible_cocina
0.0      173
1.0      211
2.0        4
3.0    12876
4.0     2355
5.0      

In [23]:
X[NUMERICAS].describe().T

,count,mean,std,min,25%,50%,75%,max
n_personas,15728.0,2.772444,1.448828,1.0,2.00,3.000000,4.0,13.0
edad_jefe,15728.0,50.211343,16.702133,16.0,37.00,49.000000,63.0,102.0
educ_jefe,15728.0,5.849377,2.855479,1.0,3.75,5.000000,8.0,13.0
n_menores15,15728.0,0.533253,0.831006,0.0,0.00,0.000000,1.0,6.0
n_mayores65,15728.0,0.358914,0.637569,0.0,0.00,0.000000,1.0,4.0
prop_dependientes,15728.0,0.308959,0.320561,0.0,0.00,0.333333,0.5,1.0
n_cuartos,15728.0,3.327505,1.079884,1.0,3.00,3.000000,4.0,41.0
n_dormitorios,15728.0,1.850839,0.815382,1.0,1.00,2.000000,2.0,7.0
prop_afiliados_salud,15728.0,0.970489,0.140352,0.0,1.00,1.000000,1.0,1.0
prop_subsidiado,15728.0,0.472385,0.449233,0.0,0.00,0.500000,1.0,1.0


In [24]:
y.value_counts().sort_index()

ses
0    3402
1    3701
2    7929
3     696
Name: count, dtype: int64

In [25]:
import sys
from ses.config import ROOT
sys.path.append(str(ROOT / "app"))
import opciones as o

In [26]:
for var, dic in o.OPCIONES.items():
    vistos = set(X[var].dropna().astype(int))
    ofrecidos = set(dic.values())
    print(var, "| faltan:", sorted(vistos - ofrecidos), "| sobran:", sorted(ofrecidos - vistos))

sexo_jefe | faltan: [] | sobran: []
educ_jefe | faltan: [] | sobran: []
tipo_vivienda | faltan: [] | sobran: []
material_paredes | faltan: [] | sobran: []
material_pisos | faltan: [] | sobran: []
estrato | faltan: [] | sobran: []
tipo_sanitario | faltan: [] | sobran: []
lugar_cocina | faltan: [] | sobran: []
combustible_cocina | faltan: [] | sobran: []
tenencia_vivienda | faltan: [] | sobran: []
posicion_ocup_jefe | faltan: [] | sobran: []


In [27]:
import joblib
from ses.pipeline import construir_pipeline
from ses.models.ordinal import POM

In [28]:
(ROOT / "models").mkdir(exist_ok=True)

for bloque in ["B", "C"]:
    modelo = construir_pipeline(POM(class_weight="balanced"), bloque=bloque)
    modelo.fit(X, y)
    joblib.dump(modelo, ROOT / "models" / f"provisional_pom_{bloque}.joblib")
    print("Modelo guardado:", bloque)

Modelo guardado: B
Modelo guardado: C


In [29]:
import importlib
import procesar as p
importlib.reload(o)
importlib.reload(p)

# Un hogar inventado: 4 personas, 2 niños, estrato 1, piso de cemento
hogar = {
    "n_personas": 4, "n_menores15": 2, "n_mayores65": 0,
    "edad_jefe": 35, "sexo_jefe": 2, "educ_jefe": 4,
    "tipo_vivienda": 1, "material_paredes": 1, "material_pisos": 2,
    "estrato": 1, "n_cuartos": 3, "n_dormitorios": 2,
    "tipo_sanitario": 1, "lugar_cocina": 1, "combustible_cocina": 4,
    "tenencia_vivienda": 3,
    "tiene_gas_natural": 0, "tiene_alcantarillado": 1,
    "tiene_recoleccion_basuras": 1, "tiene_acueducto": 1,
}

print("Errores:", p.validar(hogar))
print("Avisos:", p.avisos(hogar))

fila = p.construir_fila(hogar, bloque="B")
display(fila)

modelo_b = joblib.load(ROOT / "models" / "provisional_pom_B.joblib")
probas = modelo_b.predict_proba(fila)[0]
print("Probabilidades:", probas.round(3))
print(p.mensaje_prioridad(probas))

Errores: []
Avisos: []


,n_personas,edad_jefe,sexo_jefe,educ_jefe,n_menores15,n_mayores65,prop_dependientes,tipo_vivienda,material_paredes,material_pisos,...,tiene_alcantarillado,tiene_recoleccion_basuras,tiene_acueducto,estrato,n_cuartos,n_dormitorios,tipo_sanitario,lugar_cocina,combustible_cocina,tenencia_vivienda
0,4,35,2,4,2,0,0.5,1,1,2,...,1,1,1,1,3,2,1,1,4,3


Probabilidades: [0.874 0.106 0.018 0.002]
('alta', 'Prioridad alta: programar evaluación completa de ingreso. Probabilidad de ser Pobre o Vulnerable: 98% (clase más probable: Pobre).')
